Berikut adalah isi notebook dalam format Markdown murni dengan nama **Andrew Rizky Patulak** dan NIM **H061251027**:

# R-MET-001: Mengelompokkan Stasiun Cuaca dengan K-Means

**Andrew Rizky Patulak**

**H061251027**

Recipe ini mengikuti format **Bab 2.4 buku AI Cookbook untuk Earth Sciences**:

**Problem → Minimal Theory → Prompt → Code → Result → Optional Check**

| Keterangan | Isi |
| --- | --- |
| ID recipe | `R-MET-001` (prompt pendamping: `P-MET-001`) |
| Level | Mahasiswa meteorologi semester 3 (cukup Python dasar) |
| Library | `numpy`, `pandas`, `matplotlib`, `scikit-learn` |
| Data | Data pengamatan stasiun **sintetis** (dibuat di notebook, tidak perlu download) |
| Status eksekusi | Lihat bagian *Run record* di akhir. Status hanya "Executed" kalau sel-selnya benar-benar sudah dijalankan |

**Cara pakai:** jalankan sel dari atas ke bawah (*Run All*). Setiap bagian diberi penjelasan singkat supaya kamu tahu *kenapa* langkah itu ada, bukan hanya *apa* kodenya.

---

## 1. Problem

**Pertanyaan ilmiah:** berdasarkan karakteristik elevasi dan suhu rata-rata tahunan, bagaimana stasiun cuaca terbagi menjadi zona mikroklimat?

**Batas keputusan (penting):** hasil recipe ini hanya untuk **eksplorasi awal**. Ini bukan zonasi iklim resmi BMKG, melainkan pengelompokan berbasis jarak numerik fitur cuaca.

### Kontrak input dan constraint

| Hal | Aturan |
| --- | --- |
| Input | Tabel dengan kolom `id_stasiun`, `elevasi_m`, `suhu_rata_c` |
| Tidak boleh | Mengubah data input, menghapus baris, atau mengisi nilai hilang (cukup dilaporkan) |
| Library | Hanya numpy, pandas, matplotlib, scikit-learn. Tidak ada akses internet |
| Output | `results/r-met-001/ringkasan_klaster.csv`, `grafik_klaster.png`, dan `run_record.json` |

---

## 2. Minimal Theory

Bagian ini cukup untuk memahami kode, tidak perlu turunan rumus lengkap.

**K-Means Clustering.** Algoritma mengelompokkan data ke dalam $K$ titik pusat (*centroid*) yang ditentukan di awal. Setiap data dimasukkan ke klaster dengan jarak Euclidean terdekat:

$$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$

**Pentingnya Standardisasi (StandardScaler).** Karena elevasi berorde ratusan/ribuan meter sedangkan suhu berorde belasan/puluhan derajat Celsius, skala variabel harus disamakan terlebih dahulu (mean = 0, varians = 1) agar algoritma tidak didominasi oleh variabel berukuran besar.

**Asumsi yang dipakai:**

1. Variabel input sudah distandardisasi sebelum proses clustering.
2. Jumlah klaster ($K$) ditentukan berdasarkan kebutuhan analisis fisik lokasi.

---

## 3. Prompt (`P-MET-001`)

Prompt ini disusun dengan struktur **P = ⟨R, T, C, I, K, O, Q, V⟩**.

```text
R (Role)         : Asisten analisis data meteorologi dengan Python (pandas, scikit-learn).
T (Task)         : Kelompokkan stasiun cuaca pada [CSV_PATH] dengan K-Means,
                   jumlah klaster [N_CLUSTERS].
C (Context)      : Latihan kuliah Meteorologi Dasar. Hasil hanya untuk eksplorasi zona iklim lokal.
I (Input)        : Hanya [CSV_PATH] dengan kolom id_stasiun, elevasi_m, suhu_rata_c.
K (Constraints) : Jangan ubah file input. Jangan hapus baris atau isi nilai hilang
                   (laporkan saja). Pakai hanya numpy, pandas, matplotlib, scikit-learn.
                   Lakukan scaling fitur menggunakan StandardScaler.
O (Output)      : [OUTPUT_DIR]/ringkasan_klaster.csv, [OUTPUT_DIR]/grafik_klaster.png,
                   [OUTPUT_DIR]/run_record.json
Q (Quality)      : Jumlah baris data sama sebelum dan sesudah. Sumbu dan tabel mencantumkan satuan.
V (Verification): Bandingkan nilai mean elevasi dan suhu tiap klaster secara logis.

```

---

## 4. Code

Kodenya dibagi menjadi sel-sel kecil agar mudah dipahami satu per satu.

### 4.1 Setup dan parameter

```python
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --- parameter ---
N_CLUSTERS = 3         # Jumlah zona mikroklimat yang dicari
OUTPUT_DIR = "results/r-met-001"
CSV_PATH = os.path.join(OUTPUT_DIR, "stasiun_sintetis.csv")

RNG = np.random.default_rng(42)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Parameter siap. Output akan disimpan di:", OUTPUT_DIR)

```

### 4.2 Membuat data stasiun sintetis

```python
# Sintesis data: Elevasi berbanding terbalik dengan suhu (Lapse rate)
n_stasiun = 300
elevasi = RNG.uniform(10, 2500, n_stasiun)
suhu = 28 - (elevasi * 0.0065) + RNG.normal(0, 1.5, n_stasiun) # penurunan ~0.65C per 100m

df_sintetis = pd.DataFrame({
    "id_stasiun": [f"ST_{i+1:03d}" for i in range(n_stasiun)],
    "elevasi_m": elevasi,
    "suhu_rata_c": suhu
})

df_sintetis.to_csv(CSV_PATH, index=False)
print(f"Data sintetis disimpan: {CSV_PATH} ({len(df_sintetis)} stasiun)")
df_sintetis.head()

```

### 4.3 Memuat dan memvalidasi input

```python
data = pd.read_csv(CSV_PATH)
n_input = len(data)

cek_input = {
    "kolom_lengkap": {"id_stasiun", "elevasi_m", "suhu_rata_c"}.issubset(data.columns),
    "tidak_ada_nilai_hilang": int(data[["elevasi_m", "suhu_rata_c"]].isna().sum().sum()) == 0,
    "elevasi_valid": bool((data.elevasi_m >= 0).all()),
}
for nama, lolos in cek_input.items():
    print(("LOLOS  " if lolos else "GAGAL  ") + nama)
assert all(cek_input.values()), "Validasi input gagal!"
print(f"\nJumlah baris input: {n_input}")

```

### 4.4 Eksplorasi & Pemprosesan Clustering

```python
# Standardisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data[["elevasi_m", "suhu_rata_c"]])

# Modeling K-Means
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
data["klaster"] = kmeans.fit_predict(X_scaled)

# Visualisasi hasil
plt.figure(figsize=(8, 5))
for k in range(N_CLUSTERS):
    sub = data[data.klaster == k]
    plt.scatter(sub.elevasi_m, sub.suhu_rata_c, label=f"Klaster {k}", alpha=0.7)

plt.title("Pengelompokan Stasiun Cuaca berdasarkan Elevasi dan Suhu")
plt.xlabel("Elevasi (m)")
plt.ylabel("Suhu Rata-rata (°C)")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(OUTPUT_DIR, "grafik_klaster.png"), dpi=300)
plt.show()

```

In [12]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Tambahkan baris ini agar plot muncul di notebook
%matplotlib inline

# --- parameter ---
N_CLUSTERS = 3         # Jumlah zona mikroklimat yang dicari
OUTPUT_DIR = "results/r-met-001"
CSV_PATH = os.path.join(OUTPUT_DIR, "stasiun_sintetis.csv")

RNG = np.random.default_rng(42)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Parameter siap. Output akan disimpan di:", OUTPUT_DIR)

Parameter siap. Output akan disimpan di: results/r-met-001
